Dataset = Amazon Product Reviews Dataset
Dataset Kaggle Link = https://www.kaggle.com/datasets/yasserh/amazon-product-reviews-dataset

# Part1

In [2]:
import pandas as pd 

df = pd.read_csv('C:/Users/kumar/OneDrive/Desktop/TRY-2/Tasks-Submission/Assignment19/7817_1.csv')
df.shape
df.columns

Index(['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded',
       'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer',
       'manufacturerNumber', 'name', 'prices', 'reviews.date',
       'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating',
       'reviews.sourceURLs', 'reviews.text', 'reviews.title',
       'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes',
       'upc', 'weight'],
      dtype='object')

# Task1

1. What are word embeddings?

Word embeddings are dense numerical vectors that represent words in a continuous vector space where semantically similar words are close together.
Example:
king → [0.12, -0.45, ...]
queen → [0.11, -0.44, ...]

2. Why one-hot encoding and BoW fail?
Method	Problem
One-Hot	Huge sparse vectors, no meaning in distances
Bag of Words	Ignores word order & context, treats all words as independent

They cannot capture relationships like:
king - man + woman ≈ queen.

3. How word embeddings solve this?

Words are learned from context

Similar words get similar vectors

Captures semantic & syntactic meaning

# Part2

# Task2

What is Word2Vec?

Word2Vec is a shallow neural network model that learns word embeddings by predicting words from context.

Predicting from context

If the sentence is:

I love machine learning

Word2Vec learns that machine and learning are related because they appear together.

Definitions

Vocabulary: All unique words in the dataset

Context window: Number of words around a target

Embedding dimension: Length of each word vector (e.g., 100)

# Task3

Model	Description
CBOW	Predicts target word from context words
Skip-Gram	Predicts context words from target
When to use:

CBOW: Faster, good for large datasets

Skip-Gram: Better for rare words, smaller data

# Task4

1. Input Layer

One-hot encoded word vector

2. Hidden Layer

Weights form the word embeddings

3. Output Layer

Softmax predicts probability of context words

4. How weights become embeddings?

The trained weight matrix between input and hidden layer becomes the embedding matrix.

Input → [ W1 ] → Embedding → [ W2 ] → Output

# Part3

# Task5

In [7]:
import re

def simple_tokenizer(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

sentences = df["reviews.text"].dropna().apply(simple_tokenizer).tolist()


# Task6

In [9]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.4 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.4 MB 4.2 MB/s eta 0:00:06
   -- ------------------------------------- 1.6/24.4 MB 3.2 MB/s eta 0:00:08
   ----- ---------------------------------- 3.1/24.4 MB 5.4 MB/s eta 0:00:04
   -------- ------------------------------- 5.2/24.4 MB 6.9 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 6.9 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 6.9 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 6.9 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 6.9 MB/s eta 0:00:03
   --------- ------------------------------ 5.5/24.4 MB 2.8 MB/s eta 0:00:07
   ------------- -------------------------- 8.1/24.4 MB 3.8 MB/s eta 0:00:05
   ------------------ --------------------- 11.5/24.4 MB 5.0 MB/s eta 0:00:03
   ---------

In [10]:
from gensim.models import Word2Vec
import time
start = time.time()

cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0   # CBOW
)

cbow_time = time.time() - start
# Vocabulary size
print("CBOW Vocabulary Size:", len(cbow_model.wv.index_to_key))

# Embedding vector for a sample word
print("Vector for 'good':")
cbow_model.wv["good"][:10]   # first 10 values


CBOW Vocabulary Size: 7499
Vector for 'good':


array([ 3.0929241e-01,  1.5573607e-01, -6.0210168e-01,  2.5768465e-01,
        3.7697864e-01, -2.2579066e-01,  9.8572755e-01,  5.9144042e-04,
       -5.5469096e-01, -7.0961493e-01], dtype=float32)

# Task7

In [11]:
start = time.time()

skipgram_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1   # Skip-Gram
)

skipgram_time = time.time() - start
print("CBOW training time:", cbow_time)
print("Skip-Gram training time:", skipgram_time)


CBOW training time: 0.5076289176940918
Skip-Gram training time: 1.4039132595062256


In [12]:
cbow_model.wv.most_similar("good", topn=5)


[('surprisingly', 0.8376551270484924),
 ('buggyoperating', 0.8279632925987244),
 ('crisp', 0.8226152062416077),
 ('bass', 0.8190577626228333),
 ('represent', 0.816449761390686)]

In [13]:
skipgram_model.wv.most_similar("good", topn=5)


[('decent', 0.6653178930282593),
 ('seem', 0.6547204256057739),
 ('blu', 0.6513144373893738),
 ('streamers', 0.648765504360199),
 ('excellentauto', 0.6472480297088623)]

# Task8

In [14]:
cbow_model.wv.most_similar("product")
# Classic example (only works if words exist)
cbow_model.wv.most_similar(
    positive=["king", "woman"],
    negative=["man"]
)
cbow_model.wv.most_similar(
    positive=["price", "good"],
    negative=["bad"]
)
cbow_model.wv.similarity("good", "great")
cbow_model.wv.similarity("bad", "poor")


np.float32(0.72990024)

# Part4

# Task9

In [15]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
words = list(cbow_model.wv.index_to_key)[:50]
vectors = np.array([cbow_model.wv[word] for word in words])

pca = PCA(n_components=2)
reduced = pca.fit_transform(vectors)
plt.figure(figsize=(10,6))
plt.scatter(reduced[:,0], reduced[:,1])

for i, word in enumerate(words):
    plt.annotate(word, (reduced[i,0], reduced[i,1]))

plt.title("Word2Vec Embeddings (PCA)")
plt.show()


NameError: name 'np' is not defined

# Task10

1️ CBOW vs Skip-Gram

CBOW: Faster, better for frequent words

Skip-Gram: Slower, better for rare words

2️ Advantages of Word2Vec over TF-IDF

Captures semantic meaning

Dense vectors (not sparse)

Similar words have similar vectors

3️ Limitations of Word2Vec

No context (same word → same vector)

Needs large data

Cannot handle unseen words well

4️ Why context still matters (Transformers)

Word2Vec is static

Transformers (BERT, GPT) generate context-aware embeddings

Same word → different meaning in different sentences